In [ ]:
import json
import sys
from pathlib import Path
from time import sleep

from IPython.display import HTML, clear_output, display

sys.path.append(str(Path("../../scripts").resolve()))

from myradar import myradar
from parse_ti_mmwave_dat import parse_data

In [ ]:
CFG_PATH = Path("xwr68xx_AOP.cfg")
OUTPUT_PATH = Path("../TIradar/sit_stand_session.labeled.json")
PATTERN = [("sit", 5), ("stand_up", 5)]
REPEATS = 6
FPS = 10

In [ ]:
def build_schedule(pattern, repeats):
    return pattern * repeats

In [ ]:
schedule = build_schedule(PATTERN, REPEATS)
print(schedule)
print(sum(duration for _, duration in schedule), "seconds")

In [ ]:
def show_cue(label, seconds_left):
    display(HTML(f"<div style='font-size:48px;font-weight:700;text-align:center'>{label}</div><div style='font-size:96px;text-align:center'>{seconds_left}</div>"))

In [ ]:
def run_countdown(label, duration_s):
    for seconds_left in range(duration_s, 0, -1):
        clear_output(wait=True)
        show_cue(label, seconds_left)
        sleep(1)
    clear_output(wait=True)

In [ ]:
def capture_block(radar, cfg_path, duration_s):
    return radar.capture(str(cfg_path), duration_s)

In [ ]:
def parse_block(raw_bytes):
    return parse_data(raw_bytes)

In [ ]:
def add_block_label(frames, label, fps, block_start_s):
    labeled_frames = []
    for index, frame in enumerate(frames):
        frame["block_time_s"] = index / fps
        frame["session_time_s"] = block_start_s + index / fps
        frame["label"] = label
        labeled_frames.append(frame)
    return labeled_frames

In [ ]:
def run_session(radar, cfg_path, schedule, fps):
    merged_frames = []
    block_start_s = 0
    for block_index, (label, duration_s) in enumerate(schedule):
        run_countdown(label, duration_s)
        raw_bytes = capture_block(radar, cfg_path, duration_s)
        frames = parse_block(raw_bytes)
        labeled_frames = add_block_label(frames, label, fps, block_start_s)
        merged_frames.extend(labeled_frames)
        print(block_index, label, len(raw_bytes), len(labeled_frames))
        block_start_s += duration_s
    return merged_frames

In [ ]:
def save_labeled_frames(path, frames):
    path.write_text(json.dumps({"frames": frames}, indent=2))

In [ ]:
radar = myradar(verbose=False)
print(radar._config_port, radar._data_port)

In [ ]:
merged_frames = run_session(radar, CFG_PATH, schedule, FPS)
print(len(merged_frames))

In [ ]:
print(merged_frames[0]["label"], merged_frames[0]["session_time_s"])
print(merged_frames[-1]["label"], merged_frames[-1]["session_time_s"])

In [ ]:
save_labeled_frames(OUTPUT_PATH, merged_frames)
print(OUTPUT_PATH)

In [ ]:
def close_radar(radar):
    radar.close()

In [ ]:
close_radar(radar)
print("radar closed")